In [1]:
import torch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset

import torch.optim as optim
from annoy import AnnoyIndex
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import mlflow
import random
from mlflow.models.signature import infer_signature

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_colwidth', None)


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [5]:
# class TwoTower(nn.Module):
#     def __init__(self, product_embed_dim, query_embed_dim, hidden_dim, output_size):
#         super().__init__()
#         self.product_tower_layer1 = nn.Linear(product_embed_dim, hidden_dim)  # First hidden layer
#         self.product_tower_layer2 = nn.Linear(hidden_dim, output_size)

#         self.query_tower_layer1 = nn.Linear(query_embed_dim, hidden_dim)     # 3840*512 
#         self.query_tower_layer2 = nn.Linear(hidden_dim, output_size)
#         self.cos = nn.CosineSimilarity(dim=-1)
#         self.sigmoid = nn.Sigmoid()
#         self.relu = nn.ReLU()                

#     def forward(self, product_embed, query_embed):
#         x_p = self.product_tower_layer1(product_embed)  # Apply ReLU after first layer
#         x_p = self.relu(x_p)
#         x_p = self.product_tower_layer2(x_p)

#         x_q = self.query_tower_layer1(query_embed)  # Apply ReLU after second layer
#         x_q = self.relu(x_q)
#         x_q = self.query_tower_layer2(x_q)
        
#         x_p = F.normalize(x_p, p=2, dim=-1)
#         x_q = F.normalize(x_q, p=2, dim=-1)
#         output = self.sigmoid(self.cos(x_p, x_q))


#         return output, x_p, x_q


In [4]:
class TwoTower(nn.Module):
    def __init__(self, product_embed_dim, query_embed_dim, hidden_dim, output_size):
        super().__init__()
        self.product_tower_layer1 = nn.Linear(product_embed_dim, output_size)  # First hidden layer
        # self.product_tower_layer2 = nn.Linear(hidden_dim, output_size)

        self.query_tower_layer1 = nn.Linear(query_embed_dim, output_size)     # 3840*512 
        # self.query_tower_layer2 = nn.Linear(hidden_dim, output_size)
        self.cos = nn.CosineSimilarity(dim=-1)
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()

        # self.normalized_x_p = None
        # self.normalized_x_q = None
        
        # self.x_p = None
        # self.x_q = None

    def forward(self, product_embed, query_embed):
        x_p = self.product_tower_layer1(product_embed)  # Apply ReLU after first layer
        # x_p = self.relu(x_p)
        # x_p = self.product_tower_layer2(x_p)

        x_q = self.query_tower_layer1(query_embed)  # Apply ReLU after second layer
        # x_q = self.relu(x_q)
        # x_q = self.query_tower_layer2(x_q)
        
        normalized_x_p = F.normalize(x_p, p=2, dim=-1)
        normalized_x_q = F.normalize(x_q, p=2, dim=-1)

        # print(self.normalized_x_p)
        output = self.sigmoid(self.cos(normalized_x_p, normalized_x_q))

        # output = self.cos(normalized_x_p, normalized_x_q)


        return output, normalized_x_p, normalized_x_q


In [5]:
def reshape_array(input_array, d):
    k, _ = input_array.shape
    new_array = np.zeros((k, d))
    
    for i in range(k):
        for j in range(d):
            start_idx = j * (768 // d)
            end_idx = (j + 1) * (768 // d) if j < (d - 1) else 768
            chunk = input_array[i, start_idx:end_idx]
            new_array[i, j] = np.mean(chunk)
    
    return new_array

In [6]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy', 'query_embedding.npy']

data = []
for file in files:
    data.append(np.load(f'../data/new_embeddings/{file}'))

In [7]:
data[0] = reshape_array(data[0],32)
data[2] = reshape_array(data[2],32)
data[3] = reshape_array(data[3],32)

In [7]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4],data[5]),axis=1)

ANN search

In [6]:
files = ['product_brand_embedding.npy', 'product_bullet_point_embedding.npy', 'product_color_embedding.npy', 'product_title_embedding.npy', 'product_description_embedding.npy']

data = []
for file in files:
    data.append(np.load(f'../data/embeddings_filtered_data/{file}'))

In [7]:
embedding = np.concat((data[0],data[1],data[2],data[3],data[4]),axis=1)

In [8]:
query_embedding = np.load('../data/embeddings_filtered_data/query_embedding.npy')

In [9]:
product_title_embedding = np.load('../data/embeddings_filtered_data/product_title_embedding.npy')

In [10]:
df = pd.read_csv('../data/embeddings_filtered_data/data.csv')

In [11]:
df.iloc[2000]

Unnamed: 0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [12]:
annoy_index = AnnoyIndex(f=768, metric='euclidean')


In [13]:

for i, embed in enumerate(product_title_embedding):
    annoy_index.add_item(i, embed)

# Build the index
annoy_index.build(100) 

True

In [14]:
nearest_indices = annoy_index.get_nns_by_vector(query_embedding[0], 10,include_distances=True)
nearest_indices

([1727, 0, 62144, 88967, 36310, 52230, 94898, 28254, 34535, 48657],
 [0.49007225036621094,
  0.5528734922409058,
  0.6261216402053833,
  0.7681321501731873,
  0.7804929614067078,
  0.7851989269256592,
  0.8028589487075806,
  0.8069073557853699,
  0.8070605993270874,
  0.8071155548095703])

In [ ]:
df.iloc[nearest_indices[0]]

In [ ]:
df.iloc[nearest_indices]

Training

In [8]:
labels = pd.read_csv(f'../data/data.csv')['binary_label'].values

In [9]:
labels

array([1, 1, 1, ..., 1, 0, 1], shape=(9988,))

In [10]:
np.unique(labels, return_counts=True)

(array([0, 1]), array([2767, 7221]))

In [11]:
embedding.shape

(9988, 4608)

In [12]:
x_train, x_test, y_train, y_test = train_test_split(embedding, labels, test_size=0.2, random_state=42, shuffle=True)

In [13]:
x_train = torch.from_numpy(x_train)

x_test = torch.from_numpy(x_test)

y_train = torch.from_numpy(y_train)

y_test = torch.from_numpy(y_test)

In [14]:
y_test.unique(return_counts=True)

(tensor([0, 1]), tensor([ 547, 1451]))

In [15]:
train_dataset = TensorDataset(x_train,y_train)
test_dataset = TensorDataset(x_test,y_test)

In [16]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [17]:
# class TripletLoss(nn.Module):
#     def __init__(self, margin=1.0):
#         super(TripletLoss, self).__init__()
#         self.margin = margin
        
#     def calc_euclidean(self, x1, x2):
#         return (x1 - x2).pow(2).sum(1)
    
#     def forward(self, anchor: torch.Tensor, positive: torch.Tensor, negative: torch.Tensor) -> torch.Tensor:
#         distance_positive = self.calc_euclidean(anchor, positive)
#         distance_negative = self.calc_euclidean(anchor, negative)
#         losses = torch.relu(distance_positive - distance_negative + self.margin)

#         return losses.mean()

In [22]:
4608-768

3840

In [23]:
query_dim = 3840
product_dim = 768   

In [27]:
torch.manual_seed(42)

# def hook_fn(module, input, output):
#     print("Normalized Output:", output)

 

model = TwoTower(product_embed_dim=product_dim,query_embed_dim=query_dim, hidden_dim=16, output_size=32).to(device)

# model.product_tower_layer1.register_forward_hook(hook_fn)
# model.query_tower_layer1.register_forward_hook(hook_fn)

criterion = nn.BCELoss()
# criterion = TripletLoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

In [28]:
epochs = 100

example_product_embed = torch.rand(1, product_dim)
example_query_embed = torch.rand(1, query_dim)

mlflow.set_tracking_uri(uri='http://127.0.0.1:5050')
mlflow.start_run()
mlflow.log_param("epochs", epochs)


for epoch in range(epochs):
    total_correct = 0
    total_samples = 0
    
    model.train()
    for batch_x, batch_y in train_dataloader:
        batch_x = batch_x.to(device).float()
        batch_y = batch_y.to(device).float()
        # print(batch_x[:, :3840].shape, batch_x[:, 3840:].shape)
        
        optimizer.zero_grad()
        outputs,_,_ = model(batch_x[:, :product_dim], batch_x[:, product_dim:])
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        predictions = torch.round(outputs)
        total_correct += (predictions == batch_y).sum().item()
        total_samples += batch_y.size(0)
    
    train_accuracy = total_correct / total_samples * 100
    
    # # Log training metrics
    mlflow.log_metric("train_accuracy", train_accuracy, step=epoch+1)
    
    # Test loop
    model.eval()
    test_correct = 0
    test_samples = 0

    with torch.no_grad():
        for batch_x, batch_y in test_dataloader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float()
            
            outputs,_,_ = model(batch_x[:, :product_dim], batch_x[:, product_dim:])
            loss = criterion(outputs, batch_y)
            
            predictions = torch.round(outputs)
            test_correct += (predictions == batch_y).sum().item()
            test_samples += batch_y.size(0)
    
    test_accuracy = test_correct / test_samples * 100
    
    # Log test metrics
    mlflow.log_metric("test_accuracy", test_accuracy, step=epoch+1)
    
    model_input = torch.cat((example_product_embed, example_query_embed), dim=1).numpy()
    signature = infer_signature(model_input)

    mlflow.pytorch.log_model(model,'TwoTower',signature=signature)
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%')


mlflow.end_run()


Epoch [10/100], Train Accuracy: 73.08%, Test Accuracy: 72.27%
Epoch [20/100], Train Accuracy: 86.35%, Test Accuracy: 69.17%
Epoch [30/100], Train Accuracy: 90.64%, Test Accuracy: 68.57%
Epoch [40/100], Train Accuracy: 92.50%, Test Accuracy: 67.12%
Epoch [50/100], Train Accuracy: 93.45%, Test Accuracy: 65.07%
Epoch [60/100], Train Accuracy: 94.06%, Test Accuracy: 66.37%
Epoch [70/100], Train Accuracy: 94.31%, Test Accuracy: 66.52%
Epoch [80/100], Train Accuracy: 94.56%, Test Accuracy: 64.91%
Epoch [90/100], Train Accuracy: 94.87%, Test Accuracy: 65.32%
Epoch [100/100], Train Accuracy: 94.94%, Test Accuracy: 64.31%
🏃 View run abundant-cow-448 at: http://127.0.0.1:5050/#/experiments/0/runs/f30f249e25a14bc5a17f10ac9ba5e4fb
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


In [26]:
mlflow.end_run()

🏃 View run calm-calf-424 at: http://127.0.0.1:5050/#/experiments/0/runs/8ace7f6391b94ee7a10f26c2224abdd2
🧪 View experiment at: http://127.0.0.1:5050/#/experiments/0


In [22]:
import mlflow

loaded_model = mlflow.pytorch.load_model('../mlartifacts/0/f3e11b75caf54b55907e25835eb03e2d/artifacts/TwoTower').to(device)

In [23]:
loaded_model

TwoTower(
  (product_tower_layer1): Linear(in_features=768, out_features=32, bias=True)
  (query_tower_layer1): Linear(in_features=1632, out_features=32, bias=True)
  (cos): CosineSimilarity()
  (sigmoid): Sigmoid()
  (relu): ReLU()
)

In [27]:
np.var(embedding[:,:3840]).sum(), np.var(embedding[:,3840:]).sum() 

(np.float32(0.001300734), np.float32(0.0013007583))

In [24]:
embedding = torch.from_numpy(embedding)

In [25]:
embedding.shape

torch.Size([9988, 2400])

In [26]:
pred, normalized_x_p, normalized_x_q = loaded_model(embedding[:,:product_dim].to(device).float(),embedding[:,product_dim:].to(device).float())

In [27]:
normalized_x_p

tensor([[ 0.1595, -0.1148,  0.1381,  ...,  0.3081, -0.0919,  0.1593],
        [ 0.2436, -0.1659, -0.0261,  ..., -0.2210, -0.2048,  0.0968],
        [-0.3587,  0.0220,  0.1106,  ..., -0.1864,  0.2504, -0.3852],
        ...,
        [ 0.0923, -0.1390,  0.2741,  ...,  0.2524, -0.1978,  0.1543],
        [ 0.0200, -0.1613, -0.0476,  ...,  0.3272, -0.0749,  0.1177],
        [-0.0155, -0.1480,  0.0760,  ...,  0.1663, -0.1795, -0.0034]],
       device='cuda:0', grad_fn=<DivBackward0>)

ANN search for condensed embeddings from tower

In [28]:
annoy_index = AnnoyIndex(f=32, metric='euclidean')

In [29]:

for i, embed in enumerate(normalized_x_p):
    annoy_index.add_item(i, embed)

annoy_index.build(100,n_jobs=-1) 

True

In [30]:
nearest_indices = annoy_index.get_nns_by_vector(normalized_x_q[0], 5, include_distances=True)

In [31]:
nearest_indices

([2894, 7544, 6261, 1032, 8561],
 [0.40630850195884705,
  0.41001462936401367,
  0.4291839897632599,
  0.44951239228248596,
  0.452362596988678])

In [41]:
df = pd.read_csv('../data/data.csv')

In [20]:
df.iloc[0]

product_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [42]:
df.iloc[nearest_indices[0]]

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,query,esci_label,split,binary_label
97444,B00UAYZKL0,qorpak glc05728 clear jar with 63400 green thermoset f217 and ptfe lined cap tall straight sided 16 oz pack of 12,btl 16oz fl ss rnd 63400 grts ftef cs12,size is 69 x 169 mm\nglass material,qorpak,clear,qorpak,E,train,1
88796,B085H9VD19,new brothread 4x60 spools wooden thread rackthread holder organizer with hanging hooks for embroidery quilting and sewing threads,new brothread makes your embroidery sewing life easier and organized set of 4x60 spools wooden thread racks for holding maximum 240 mini spools and conesnot suitable for large spools new brothread thread rack is perfectly suitable for all the new brothreads 500m and 1000m spool threads and other thread sellers on amazon only when the max base size of the spool is larger than 170 inch such as 1000m spools of floriani isacord robisonanton you need put 23 spools upside down each row on the rack two folded legs can be easily unfolded and stand freely on the table fold the legs back and mount it on the wall using two hooks provided on back of the rack made of nature wood with smooth surface you can diy to paint or stain your thread rack as you like available for organize bobbins rings and other small sewing and embroidery articles,set of 4x60 mini spools thread racks this package includes four sets of 60 spools thread racks shocking price of the single rack in this package can compete with the lowest price sold in supermarkets this is also the best and economical package to share 4 sets of wooden thread racks with your family neighbors and friends\n make your embroidery and quilting easierthe center distance of two spindles is 17 inch the height of each spindle is 14 inchwhich is designed for holding 60 mini king spools cones not suitable for large spools it is a great helper to nicely organize your thread spools and make your sewing room or sewing desk to be clean and well organized displaying all the color assortment of your threads you can easily see and reach each spool when making your embroidery and quilting pro\n can be free standing or mounted on the wall these wooden thread racks are designed with two folded legs you can easily pull the legs out and let it stand freely on your table or fold the legs back for easy storage in order to saving space we have designed two hooks on back of the rack you can hang this thread rack on the wall note nails on the wall and other tools are not provided\n widely used for most mini king spools and cones these wooden thread racks are perfectly suitable for all the new brothreads 500m and 1000m spool threads and other thread sellers on amazon only when the max base size of the spool is larger than 170 inch such as 1000m spools of floriani isacord robisonanton you need put 23 spools upside down each row on the rack\n premium natural wood and painted allowed these 60 spools wooden thread racks are made of premium natural beech wood with smooth finishing it can be diy painted or stained to match your embroidery and sewing machines or your sewing room and sewing desk note the threads on the pictures dont come with the wooden rack new brothread always think what you think do what you do love what you love,new brothread,4 x 60 spools,thread spool holder,E,train,1
83782,B074WYRJX8,40 pack clothing rack size dividers wardrobe round hangers dividers with 1 piece marker pen,specifications material sturdy plastic overall diameter 35 inch 9 cm inside diameter 1375 inch 35 cm marker pen nonwashable features 40 pieces rack size dividers allow you to display and sort your objects through information such as size color store logo holds new items etc hole diameter on divider is 1375 inch these size dividers work with rods with a diameter of 1375 inch or less black marker pen is nonwashable note please make sure your closet rod is small enough for these size dividers to go on package including 40 size di

In [42]:
cos = nn.CosineSimilarity(dim=-1)
cos(loaded_model.normalized_x_p[19326], loaded_model.normalized_x_q[0])


tensor(0.9671, device='cuda:0', grad_fn=<SumBackward1>)